In [ ]:
#| default_exp pool

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *

Keep one kernel per key and enforce a shared kernel limit.

In [ ]:
#| export
from __future__ import annotations
import asyncio, os, time
from pathlib import Path
from fastcore.all import L, first, ifnone
from kunda.kernel import GatewayKernel, GatewayService, Kernel
from kunda.kernel import KERNELS
from kunda.spec import KernelStartError, kernelspec_for
from kunda.pythons import app_env, env_name

In [ ]:
#| export
#: How long a kernel may sit idle before the sweeper closes it. `$<prefix>KERNEL_IDLE` overrides.
IDLE_SECONDS, SWEEP_SECONDS = 30*60, 60

The host-prefixed `KERNEL_IDLE` environment variable overrides `IDLE_SECONDS`.

In [ ]:
#| export
class KernelLimit(RuntimeError):
    "The ceiling is reached. `candidate` is a kernel that can be offered to close, or None while all are busy."
    def __init__(self, msg, candidate=None, limit=0):
        super().__init__(msg)
        self.candidate, self.limit = candidate, limit

`KernelLimit` reports the configured limit and an optional kernel that could be closed.

In [ ]:
#| export
class RuntimeBroker:
    "A ceiling across every pool in this process. Never evicts a namespace somebody is using."
    def __init__(self, max_kernels=None, auto_manage=None):
        self.max_kernels = int(max_kernels or app_env('MAX_KERNELS', 12))
        self.auto_manage = bool(ifnone(auto_manage, app_env('KERNEL_AUTO', '').lower() in ('1', 'true', 'yes')))
        self.pools, self._lock, self._reserved = [], asyncio.Lock(), set()
    def register(self, pool):
        if pool not in self.pools: self.pools.append(pool)
        pool.broker = self
        return pool
    @property
    def live(self):
        return [(pool, key, kernel) for pool in self.pools for key, kernel in pool.kernels.items()
            if kernel.alive]
    def status(self):
        return {'live': len(self.live), 'limit': self.max_kernels, 'runtimes': [{'key': str(key), 'pid': kernel.pid,
                     'kind': kernel.kernel_kind, 'cwd': str(kernel.cwd or '')} for _, key, kernel in self.live]}
    def stalest(self):
        "The idle kernel that has waited longest, as `(pool, key, kernel)`. None while all are busy."
        idle = [row for row in self.live if not row[2].busy]
        return max(idle, key=lambda row: row[2].idle_for) if idle else None
    def _candidate(self):
        "What the limit dialog needs to name the kernel it is offering to close."
        if (row := self.stalest()) is None: return None
        _, key, k = row
        return {'key': str(key), 'name': Path(str(getattr(k, 'cwd', '') or key)).name, 'cwd': str(getattr(k, 'cwd', '') or ''),
                'pid': getattr(k, 'pid', None), 'idle_for': int(k.idle_for)}
    async def admit(self, pool, key):
        token, evicted = (id(pool), key), None
        async with self._lock:
            if token in self._reserved or key in pool._starting: return
            if len(self.live) + len(self._reserved) >= self.max_kernels:
                row = self.stalest()
                if row is None: raise KernelLimit(f'kernel limit reached ({self.max_kernels}) and every runtime is busy; stop one '
                    f'or set {env_name("MAX_KERNELS")} to a larger value', limit=self.max_kernels)
                if not self.auto_manage: raise KernelLimit(f'kernel limit reached ({self.max_kernels}); close an idle runtime '
                    f'or set {env_name("MAX_KERNELS")} to a larger value',
                    candidate=self._candidate(), limit=self.max_kernels)
                evicted = self._candidate()
                vpool, vkey, _ = row
                await vpool.close(vkey)
            self._reserved.add(token)
        return evicted
    async def release(self, pool, key):
        async with self._lock: self._reserved.discard((id(pool), key))

`RuntimeBroker` counts live kernels across registered pools. `admit` raises `KernelLimit` when the next kernel would exceed `max_kernels`.

In [ ]:
#| export
class KernelPool:
    "Live kernels, keyed by an id the host assigns: usually a tab, a notebook or a folder."
    def __init__(self, port=8000, default_kernel='ipykernel', default_python=None, broker=None,
        transport='direct', gateway=None, idle=None, runner_for=None, known_kernels=None,
        install_hints=None):
        self.runner_for, self.known_kernels = runner_for, known_kernels or {}
        self.install_hints = install_hints or {}
        self.port, self.kernels, self._starting, self.broker = port, {}, {}, None
        self.idle = int(ifnone(idle, app_env('KERNEL_IDLE', IDLE_SECONDS)))
        self._sweep = None
        self.transport = transport if transport in ('direct', 'gateway') else 'direct'
        self.gateway = gateway
        if broker is not None: broker.register(self)
        self.default_kernel = default_kernel if default_kernel in KERNELS else 'ipykernel'
        self.default_python = default_python
    def choose(self,kernel=None,python=False):
        "Set what the *next* kernel starts as. Running kernels are left alone."
        if kernel and kernel in KERNELS: self.default_kernel = kernel
        if python is not False: self.default_python = python
        return {'kernel': self.default_kernel, 'python': self.default_python}
    def _class_for(self, kw):
        "Select the Jupyter kernel or host runner for a language."
        lang = kw.get('lang') or 'python'
        if lang == 'python': return GatewayKernel if self.transport == 'gateway' else Kernel
        if kernelspec_for(lang, self.known_kernels): return Kernel
        if self.runner_for and (r := self.runner_for(lang)) is not None: return r
        return Kernel
    async def get(self, key, **kw):
        "The kernel for `key`, starting it once even when several browser panels ask together."
        self._sweeping()
        if (task := self._starting.get(key)) is not None: return await asyncio.shield(task)
        if (k := self.kernels.get(key)) is not None and k.alive: return k.touch()
        evicted = await self.broker.admit(self, key) if self.broker is not None else None
        async def start_one():
            kw.setdefault('kernel', self.default_kernel)
            kw.setdefault('python', self.default_python)
            cls, args = self._class_for(kw), dict(kw)
            if cls is GatewayKernel:
                if self.gateway is None: self.gateway = GatewayService().start()
                args['gateway'] = self.gateway
                args.pop('lang', None)
            if issubclass(cls, Kernel):
                args['known'] = self.known_kernels
                args['install'] = self.install_hints.get(args.get('lang') or 'python', '')
            if getattr(cls, 'wants_key', False): args['key'] = key
            k = self.kernels[key] = cls(port=self.port, **args)
            k.evicted = evicted
            k.touch()
            try: return await k.start()
            except BaseException:
                if self.kernels.get(key) is k: self.kernels.pop(key, None)
                raise
        task = self._starting.get(key)
        if task is None: task = self._starting[key] = asyncio.create_task(start_one())
        try:
            return await asyncio.shield(task)
        finally:
            if task.done() and self._starting.get(key) is task:
                self._starting.pop(key, None)
                if self.broker is not None: await self.broker.release(self, key)
    async def reap(self):
        "Close what nobody has touched for `idle` seconds. A running cell is never taken."
        if self.idle <= 0: return []
        stale = [k for k, v in self.kernels.items()
                 if k not in self._starting and not v.busy and v.idle_for > self.idle]
        for k in stale: await self.close(k)
        return stale
    def _sweeping(self):
        "Start the idle sweep on first use: `get` is the only thing that ever adds a kernel."
        if self.idle > 0 and (self._sweep is None or self._sweep.done()):
            self._sweep = asyncio.create_task(self._sweep_loop())
    async def _sweep_loop(self):
        while True:
            await asyncio.sleep(min(SWEEP_SECONDS, max(1, self.idle)))
            try: await self.reap()
            except asyncio.CancelledError: raise
            except Exception: pass
    def peek(self, key):
        "A completed live kernel, never the object currently being started."
        if key in self._starting: return None
        k = self.kernels.get(key)
        return k if k is not None and k.alive else None
    async def close(self, key):
        if (task := self._starting.get(key)) is not None:
            try: await asyncio.shield(task)
            except BaseException: pass
        if (k := self.kernels.pop(key, None)) is not None: await k.shutdown()
    async def close_all(self):
        if self._sweep is not None: self._sweep.cancel(); self._sweep = None
        tasks = list(self._starting.values())
        if tasks: await asyncio.gather(*[asyncio.shield(task) for task in tasks], return_exceptions=True)
        await asyncio.gather(*[k.shutdown() for k in self.kernels.values()], return_exceptions=True)
        self.kernels.clear()

In [ ]:
#| hide
#| exec_doc
class Fake:
    "A kernel-shaped stand-in: what the pool and the broker read, and a shutdown that answers."
    alive, pid, kernel_kind = True, 4242, 'ipykernel'
    def __init__(self, cwd, idle_for=0., busy=False): self.cwd, self.idle_for, self.busy = cwd, idle_for, busy
    async def shutdown(self): self.alive = False

`KernelPool.get` starts at most one kernel for each key. Concurrent callers wait for the same start. Failed starts are not stored.

In [ ]:
#| exec_doc
p = KernelPool()
p.choose(kernel='ipymini', python='/proj/.venv/bin/python')

In [ ]:
#| hide
test_eq(p.choose(kernel='nonsense')['kernel'], 'ipymini')     # an unknown name changes nothing
test_eq(p.choose()['python'], '/proj/.venv/bin/python')       # and not passing it is not clearing it
test_is(p.choose(python=None)['python'], None)                # passing None is

`_class_for` selects the runner for a language and transport. The gateway transport supports Python only.

In [ ]:
#| exec_doc
gw = KernelPool(transport='gateway')
(gw._class_for({'lang': 'python'}).__name__, gw._class_for({'lang': 'rust'}).__name__,
 KernelPool(transport='carrier-pigeon').transport)

`reap` closes kernels idle for at least `idle` seconds. Busy kernels are kept. `idle=0` disables reaping.

In [ ]:
#| exec_doc
pool = KernelPool(idle=60)
pool.kernels.update(stale=Fake('/proj/stale', idle_for=90), fresh=Fake('/proj/fresh', idle_for=5),
                    working=Fake('/proj/working', idle_for=900, busy=True))
await pool.reap(), list(pool.kernels)

In [ ]:
#| hide
test_eq(pool.kernels['fresh'].alive, True)
pool.idle = 0
test_eq(await pool.reap(), [])                # switched off, and the same kernels are still stale
test_is(pool.peek('working'), pool.kernels['working'])
test_is(pool.peek('gone'), None)
await pool.close_all()
test_eq(pool.kernels, {})

The examples use stand-in kernels and do not start child processes.

In [ ]:
#| exec_doc
b = RuntimeBroker(max_kernels=2, auto_manage=False)
pool = KernelPool(idle=0, broker=b)
pool.kernels.update(one=Fake('/proj/one', idle_for=90), two=Fake('/proj/two', busy=True))
b.status()

`status` lists current kernels. `_candidate` selects the stalest idle kernel that could be closed.

In [ ]:
#| exec_doc
b.stalest()[1], b._candidate()

Without automatic management, `admit` reports a candidate and closes nothing.

In [ ]:
#| exec_doc
try: await b.admit(pool, 'three')
except KernelLimit as e: print(e); print(e.candidate['name'], e.limit)

When every kernel is busy, `candidate` is `None`.

In [ ]:
#| exec_doc
pool.kernels['one'].busy = True
try: await b.admit(pool, 'three')
except KernelLimit as e: print(e); print(e.candidate, b.stalest())

With `auto_manage=True`, `admit` closes the stalest idle kernel and returns its description.

In [ ]:
#| exec_doc
auto = RuntimeBroker(max_kernels=1, auto_manage=True)
p2 = KernelPool(idle=0, broker=auto)
p2.kernels['one'] = Fake('/proj/one', idle_for=90)
evicted = await auto.admit(p2, 'two')
evicted['key'], list(p2.kernels), auto.status()['live']

In [ ]:
#| hide
test_eq(auto._reserved, {(id(p2), 'two')})    # nothing is alive yet, and the slot is held anyway
await auto.release(p2, 'two')
test_eq(auto._reserved, set())
test_is(await auto.admit(p2, 'two'), None)    # room again, so nothing was evicted to make it
await auto.release(p2, 'two')
test_eq(b.pools, [pool]); test_is(pool.broker, b)
await pool.close_all(); await p2.close_all()

The pool passes `known_kernels` and `install_hints` to local Kunda kernels.

In [ ]:
#| exec_doc
p3 = KernelPool(idle=0, known_kernels={'rust': 'evcxr'}, install_hints={'rust': 'cargo install evcxr_jupyter'})
err = ''
try: await p3.get('rs', lang='rust')
except KernelStartError as e: err = str(e)
err

In [ ]:
#| hide
test_eq(err, 'no Jupyter kernel is installed for rust. Install one with `cargo install evcxr_jupyter`.')
test_eq(list(p3.kernels), [])                       # a start that raised leaves no kernel behind
test_eq(KernelPool().install_hints, {})
await p3.close_all()

Pool and broker environment variables use the host prefix set by `use_app`.

In [ ]:
#| hide
from kunda.pythons import use_app
use_app('leela', 'LEELA_')
os.environ.update(LEELA_MAX_KERNELS='1', LEELA_KERNEL_AUTO='yes', LEELA_KERNEL_IDLE='5')
b3 = RuntimeBroker()
test_eq((b3.max_kernels, b3.auto_manage, KernelPool().idle), (1, True, 5))
p4 = KernelPool(idle=0, broker=RuntimeBroker(max_kernels=1, auto_manage=False))
p4.kernels['one'] = Fake('/proj/one', idle_for=90)
err = ''
try: await p4.broker.admit(p4, 'two')
except KernelLimit as e: err = str(e)
test_eq(err, 'kernel limit reached (1); close an idle runtime or set LEELA_MAX_KERNELS to a larger value')
p4.kernels['one'].busy = True
try: await p4.broker.admit(p4, 'two')
except KernelLimit as e: err = str(e)
test_eq(err, 'kernel limit reached (1) and every runtime is busy; stop one or set LEELA_MAX_KERNELS to a larger value')
for k in ('LEELA_MAX_KERNELS', 'LEELA_KERNEL_AUTO', 'LEELA_KERNEL_IDLE'): del os.environ[k]
use_app()
test_eq((RuntimeBroker().max_kernels, RuntimeBroker().auto_manage, KernelPool().idle),
        (12, False, IDLE_SECONDS))
await p4.close_all()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()